In [0]:
# Initialize the set up
import sqlite3
import pandas as pd
import os

# Your exact path string pointing directly to the file inside your volume
volume_file_path = "/Volumes/workspace/default/ds625_tp/cityu.db"

# Direct validation check to ensure Databricks sees the file
if not os.path.exists(volume_file_path):
    raise FileNotFoundError(
        f"File not found at: '{volume_file_path}'\n"
        "Please confirm that the file is named exactly 'cityu.db' inside your ds625_tp volume."
    )

print(f" File located successfully! Size: {os.path.getsize(volume_file_path)} bytes.")

try:
    # Open connection using standard Read-Only URI parameters to prevent cloud locks
    conn = sqlite3.connect(f"file:{volume_file_path}?mode=ro", uri=True)
    cursor = conn.cursor()

    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    print(cursor.fetchall())
    # Read your SQLite scraped data tables into memory frames
    courses_pd = pd.read_sql_query("SELECT * FROM courses;", conn)
    prereqs_pd = pd.read_sql_query("SELECT * FROM prerequisites;", conn)
    conn.close()
    
    # Transform raw data frames into Spark cloud distributed dataframes
    spark_courses = spark.createDataFrame(courses_pd)
    spark_prereqs = spark.createDataFrame(prereqs_pd)
    
    # Save directly out as high-performance permanent Delta Tables
    spark_courses.write.format("delta").mode("overwrite").saveAsTable("workspace.default.catalog_courses")
    spark_prereqs.write.format("delta").mode("overwrite").saveAsTable("workspace.default.catalog_explicit_prerequisites")
    
    print("\n Success! Delta Tables are now live inside 'workspace.default':")
    print("  • workspace.default.catalog_courses")
    print("  • workspace.default.catalog_explicit_prerequisites")

except Exception as e:
    print(f"Ingestion Error: {e}")


 File located successfully! Size: 745472 bytes.
[('courses',), ('sqlite_sequence',), ('prerequisites',), ('degree_requirements',), ('faqs',)]

 Success! Delta Tables are now live inside 'workspace.default':
  • workspace.default.catalog_courses
  • workspace.default.catalog_explicit_prerequisites


In [0]:
%sql
-- Query 1: Summary metrics of course requirements
-- Calculates exactly how many courses have requirements vs. those that do not

SELECT 
    CASE 
        WHEN p.course_code IS NOT NULL THEN 'Has Prerequisites'
        ELSE 'No Prerequisites Mentioned'
    END AS `Status`,
    COUNT(DISTINCT c.code) AS `Total Courses`,
    ROUND(COUNT(DISTINCT c.code) * 100.0 / (SELECT COUNT(*) FROM workspace.default.catalog_courses), 1) AS `Percentage of Catalog`
FROM workspace.default.catalog_courses c
LEFT JOIN workspace.default.catalog_explicit_prerequisites p ON c.code = p.course_code
GROUP BY `Status`
ORDER BY `Total Courses` ASC;


Status,Total Courses,Percentage of Catalog
Has Prerequisites,201,21.3
No Prerequisites Mentioned,744,78.7


In [0]:
%sql
-- Query 2: Categorize all courses by prerequisite presence
-- Compiles explicit catalog requirements using distributed Spark SQL aggregations

SELECT 
    c.code AS `Course Code`,
    c.title AS `Course Title`,
    c.credits AS `Credits`,
    CASE 
        WHEN p.course_code IS NOT NULL THEN 'Has Prerequisites'
        ELSE 'No Prerequisites Mentioned'
    END AS `Prerequisite Status`,
    IFNULL(CONCAT_WS(', ', COLLECT_LIST(p.prereq_code)), 'None') AS `Prerequisite List`
FROM workspace.default.catalog_courses c
LEFT JOIN workspace.default.catalog_explicit_prerequisites p ON c.code = p.course_code
GROUP BY c.code, c.title, c.credits, p.course_code
ORDER BY `Prerequisite Status` DESC, c.code ASC;


Course Code,Course Title,Credits,Prerequisite Status,Prerequisite List
AC215,AC 215 Fundamentals of Accounting,5,No Prerequisites Mentioned,
AC501,AC 501 Applied Management Accounting Concepts I,3,No Prerequisites Mentioned,
AC502,AC 502 Applied Management Accounting Concepts II,3,No Prerequisites Mentioned,
AC530,AC 530 CPA Review - Financial Accounting & Reporting (FAR),3,No Prerequisites Mentioned,
AC531,AC 531 CPA Review - Regulation (REG),3,No Prerequisites Mentioned,
AC532,AC 532 CPA Review - Auditing & Attestation (AUD),3,No Prerequisites Mentioned,
AC533,AC 533 CPA Review - Business Environment & Concepts (BEC),3,No Prerequisites Mentioned,
AC540,AC 540 Auditing Techniques,3,No Prerequisites Mentioned,
AC550,AC 550 Auditing Theory and Practice,3,No Prerequisites Mentioned,
AI500,AI 500 Artificial Intelligence,3,No Prerequisites Mentioned,


In [0]:
%sql
---Query 3: Audit Credit Weight Distribution by Academic Department
---This query strips out the alphabetic prefixes from your scraped course codes to calculate credit weights and curriculum density across different ---academic departments:
SELECT 
    REGEXP_EXTRACT(code, '^([A-Z]+)', 1) AS `Academic Department`,
    COUNT(code) AS `Total Scraped Courses`,
    ROUND(AVG(credits), 2) AS `Average Credit Weight`,
    SUM(credits) AS `Total Departmental Units`
FROM workspace.default.catalog_courses
GROUP BY `Academic Department`
ORDER BY `Total Scraped Courses` DESC;


Academic Department,Total Scraped Courses,Average Credit Weight,Total Departmental Units
CS,68,4.15,282
EEA,44,2.7,119
ETC,39,2.41,94
COUN,37,3.73,138
CPC,36,3.08,111
PSY,35,3.0,105
IS,35,4.89,171
MBA,35,3.0,105
EGC,27,2.37,64
PM,25,3.0,75


In [0]:
%sql
-- Query 4: Find the "Bottleneck" Core Courses
-- This query identifies which courses are prerequisites for the highest number of other classes. 
-- These are the most critical milestone courses in the university curriculum:

SELECT 
    p.prereq_code AS `Prerequisite Course`, 
    c.title AS `Course Title`,
    COUNT(p.course_code) AS `Number of Dependent Courses`
FROM workspace.default.catalog_explicit_prerequisites p
JOIN workspace.default.catalog_courses c ON p.prereq_code = c.code
GROUP BY p.prereq_code, c.title
ORDER BY `Number of Dependent Courses` DESC
LIMIT 10;


Prerequisite Course,Course Title,Number of Dependent Courses
IS360,IS 360 Database Technologies,13
PSY201,PSY 201 Introduction to Psychology (SS),12
MBA500,MBA 500 Essentials of Business Management,11
PSY202,PSY 202 Understanding Human Development (SS),11
PSY209,PSY 209 Fundamentals of Research Methods in Social Sciences (NS),10
CPC514,CPC 514 Research Methods and Statistics,10
PSY240,PSY 240 Critical Thinking and Writing Skills in Social Sciences (HU),10
MBA501,MBA 501 Global Business Communication and Research,9
IS201,IS 201 Fundamentals of Computing,9
TESOL510,TESOL 510 Principles of Language Learning and Teaching,8
